# Aggregate the complete verified P4b matrix

Use a CPU session with Internet enabled. Attach all 15 exact private shard datasets v1, `thestonedape/task-aware-eeg2text-p4b-full-shard-verifications` v1, task-segmented protocol v1, task-segmented schedule v1, and the exact launch-authorization dataset. Enable `GITHUB_TOKEN`, `P4B_FULL_LAUNCH_SHA256`, and the out-of-band `P4B_FROZEN_REGISTRY_SHA256`. This is the only notebook that can promote scientific-decision permission, and only after all 15 shards are independently reverified against the frozen registry.

In [ ]:
import glob, hashlib, json, os, platform, re, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
import numpy as np
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
WORKTREE = Path('/kaggle/working/SemKey')
ASKPASS = Path('/kaggle/working/git_askpass.py')
OUTPUT = Path('/kaggle/working/task-aware-eeg2text-p4b-full-matrix')
EXPECTED_UNITS = [(fold, seed) for fold in range(5) for seed in (20260717, 20260718, 20260719)]
EXPECTED_SHARDS = [f'p4b-f{fold}-s{seed}' for fold, seed in EXPECTED_UNITS]
PIN_PATHS = {
    'runner_source_sha256': 'evaluation/run_task_segmented_full_shard.py',
    'adapter_source_sha256': 'project_adapters/task_segmented_objective.py',
    'task_treatment_pilots_source_sha256': 'project_adapters/task_treatment_pilots.py',
    'shard_verifier_source_sha256': 'evaluation/verify_task_segmented_full_shard_artifact.py',
    'aggregator_source_sha256': 'evaluation/aggregate_task_segmented_full_shards.py',
    'decision_engine_source_sha256': 'evaluation/decide_task_segmented_objective.py',
    'execution_notebook_sha256': 'kaggle/run_task_segmented_full_shard.ipynb',
    'shard_clean_remount_verification_notebook_sha256': 'kaggle/verify_task_segmented_full_shard_artifact.ipynb',
    'complete_matrix_aggregation_notebook_sha256': 'kaggle/aggregate_task_segmented_full_shards.ipynb',
}

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

def is_sha256(value):
    return isinstance(value, str) and re.fullmatch(r'[0-9a-f]{64}', value) is not None

secrets = UserSecretsClient()
def required_secret(name):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    assert value and value.strip(), f'Enable the private Kaggle secret {name}'
    return value.strip()

LAUNCH_SHA256 = required_secret('P4B_FULL_LAUNCH_SHA256').lower()
REGISTRY_SHA256 = required_secret('P4B_FROZEN_REGISTRY_SHA256').lower()
assert is_sha256(LAUNCH_SHA256) and is_sha256(REGISTRY_SHA256)
launch_candidates = sorted(set(glob.glob(
    '/kaggle/input/**/task_segmented_full_launch_authorization.json', recursive=True
)))
assert len(launch_candidates) == 1, ('Attach exactly one launch authorization dataset', launch_candidates)
LAUNCH_PATH = Path(launch_candidates[0])
assert LAUNCH_PATH.is_file() and not LAUNCH_PATH.is_symlink() and digest(LAUNCH_PATH) == LAUNCH_SHA256
launch = json.loads(LAUNCH_PATH.read_text(encoding='utf-8'))
base_fields = {
    'schema_version', 'status', 'full_execution_contract_sha256', 'project_commit',
    'runtime_environment', 'authorized_shard_ids', 'full_training_authorized',
    'checkpoint_evaluation_authorized', 'confirmation_evaluation_authorized',
    'scientific_decision_permitted_after_complete_matrix_only',
    'partial_result_scientific_inspection_permitted',
    'official_validation_rows_read', 'official_validation_used_for_confirmation',
    'held_out_test_rows_read', 'held_out_test_accessed',
}
assert set(launch) == base_fields | set(PIN_PATHS)
assert launch['schema_version'] == 1 and launch['status'] == 'authorized_for_full_p4b_launch'
assert launch['authorized_shard_ids'] == EXPECTED_SHARDS
assert launch['runtime_environment'] == {
    'python': '3.12.13', 'numpy': '2.0.2', 'torch': '2.10.0+cu128',
    'torch_cuda': '12.8', 'device': 'cuda:0', 'minimum_cuda_device_count': 1,
    'selected_cuda_device_index': 0, 'selected_cuda_device_name': 'Tesla T4',
    'selected_cuda_compute_capability': [7, 5], 'cublas_workspace_config': ':4096:8',
    'deterministic_algorithms_required': True,
    'full_scientific_cpu_execution_permitted': False,
    'runtime_fingerprint_bound_to_shard_and_resume': True,
}
assert platform.python_version() == launch['runtime_environment']['python']
assert np.__version__ == launch['runtime_environment']['numpy']
assert launch['full_training_authorized'] is True
assert launch['checkpoint_evaluation_authorized'] is True and launch['confirmation_evaluation_authorized'] is True
assert launch['scientific_decision_permitted_after_complete_matrix_only'] is True
for field in ('partial_result_scientific_inspection_permitted', 'official_validation_rows_read', 'official_validation_used_for_confirmation', 'held_out_test_rows_read', 'held_out_test_accessed'):
    assert launch[field] is False, field
PROJECT_COMMIT = launch['project_commit']
assert isinstance(PROJECT_COMMIT, str) and re.fullmatch(r'[0-9a-f]{40}', PROJECT_COMMIT)
assert is_sha256(launch['full_execution_contract_sha256']) and all(is_sha256(launch[key]) for key in PIN_PATHS)


In [ ]:
github_token = required_secret('GITHUB_TOKEN')
if WORKTREE.exists():
    shutil.rmtree(WORKTREE)
ASKPASS.write_text(
    "#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n",
    encoding='utf-8', newline='\n',
)
os.chmod(ASKPASS, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': str(ASKPASS), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token, 'PYTHONDONTWRITEBYTECODE': '1'})
try:
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(WORKTREE)], check=True, env=clone_env)
finally:
    if ASKPASS.exists():
        ASKPASS.unlink()
    del github_token, clone_env
subprocess.run(['git', '-C', str(WORKTREE), 'checkout', '--detach', PROJECT_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(WORKTREE), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PROJECT_COMMIT
assert digest(WORKTREE / 'evaluation/task_segmented_full_execution_contract.json') == launch['full_execution_contract_sha256']
for key, relative in PIN_PATHS.items():
    path = WORKTREE / relative
    assert path.is_file() and not path.is_symlink() and digest(path) == launch[key], key

def assert_clean_git():
    status = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'status', '--porcelain=v1', '--untracked-files=all', '--ignored=matching'], text=True
    ).strip()
    assert status == '', ('Git worktree is not clean', status)
    submodules = subprocess.check_output(
        ['git', '-C', str(WORKTREE), 'submodule', 'status', '--recursive'], text=True
    ).splitlines()
    assert all(line.startswith(' ') for line in submodules), ('Git submodule drift', submodules)

assert_clean_git()
test_env = os.environ.copy()
test_env['PYTHONDONTWRITEBYTECODE'] = '1'
subprocess.run([
    sys.executable, '-B', '-m', 'unittest',
    'evaluation.test_verify_task_segmented_full_shard_artifact',
    'evaluation.test_aggregate_task_segmented_full_shards',
    'evaluation.test_decide_task_segmented_objective',
], check=True, cwd=WORKTREE, env=test_env)
assert_clean_git()
print({'project_commit': actual_commit, 'local_launch_pins': 'PASS', 'regressions': 'PASS', 'git_clean': True})


In [ ]:
PROTOCOL_REQUIRED = {
    'batch_grid_feasibility.csv', 'candidate_pools.csv', 'confirmation_donors.csv',
    'outer_split_assignments.csv', 'protocol_registry.json', 'pseudo_groups.csv',
    'text_group_folds.csv', 'task_segmented_protocol_report.json',
    'protocol_freeze_run_metadata.json', 'task_segmented_objective_contract.json',
}
SCHEDULE_REQUIRED = {
    'trial_catalog.csv', 'schedule_indices.u32le', 'schedule_units.csv',
    'schedule_audit.csv', 'task_segmented_training_schedule_manifest.json',
    'task_segmented_training_schedule_report.json',
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json', 'schedule_freeze_run_metadata.json',
}
def exact_roots(marker, required):
    roots = []
    for marker_path in glob.glob('/kaggle/input/**/' + marker, recursive=True):
        root = Path(marker_path).parent
        try:
            names = {path.name for path in root.iterdir()}
        except OSError:
            continue
        if names == required and not root.is_symlink() and all(not path.is_symlink() for path in root.iterdir()):
            roots.append(root)
    return sorted(set(roots))
protocol_roots = exact_roots('task_segmented_protocol_report.json', PROTOCOL_REQUIRED)
schedule_roots = exact_roots('schedule_freeze_run_metadata.json', SCHEDULE_REQUIRED)
assert len(protocol_roots) == len(schedule_roots) == 1
PROTOCOL_ROOT, SCHEDULE_ROOT = protocol_roots[0], schedule_roots[0]
assert 'task-aware-eeg2text-task-segmented-protocol' in PROTOCOL_ROOT.parts
assert 'task-aware-eeg2text-task-segmented-schedule' in SCHEDULE_ROOT.parts

registry_candidates = sorted(set(glob.glob('/kaggle/input/**/frozen_shard_registry.json', recursive=True)))
assert len(registry_candidates) == 1, ('Attach exact verification/registry dataset Version 1', registry_candidates)
REGISTRY_PATH = Path(registry_candidates[0])
REGISTRY_ROOT = REGISTRY_PATH.parent
assert 'task-aware-eeg2text-p4b-full-shard-verifications' in REGISTRY_ROOT.parts
assert REGISTRY_PATH.is_file() and not REGISTRY_PATH.is_symlink() and digest(REGISTRY_PATH) == REGISTRY_SHA256
registry = json.loads(REGISTRY_PATH.read_text(encoding='utf-8'))
assert set(registry) == {
    'schema_version', 'status', 'full_execution_contract_sha256',
    'launch_authorization_sha256', 'partial_scientific_decision_permitted',
    'held_out_test_accessed', 'shards',
}
assert registry['schema_version'] == 1 and registry['status'] == 'frozen_after_all_p4b_shards_preserved'
assert registry['full_execution_contract_sha256'] == launch['full_execution_contract_sha256']
assert registry['launch_authorization_sha256'] == LAUNCH_SHA256
assert registry['partial_scientific_decision_permitted'] is False and registry['held_out_test_accessed'] is False
assert len(registry['shards']) == 15

shard_roots = {}
for manifest_path in glob.glob('/kaggle/input/**/full_shard_manifest.json', recursive=True):
    path = Path(manifest_path)
    if path.is_symlink():
        continue
    try:
        manifest = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, UnicodeDecodeError, json.JSONDecodeError):
        continue
    shard_id = manifest.get('shard_id')
    if shard_id not in EXPECTED_SHARDS:
        continue
    root = path.parent
    assert f'task-aware-eeg2text-{shard_id}' in root.parts
    assert shard_id not in shard_roots, ('Duplicate shard dataset', shard_id)
    shard_roots[shard_id] = root
assert set(shard_roots) == set(EXPECTED_SHARDS)

report_by_shard = {}
for entry, unit in zip(registry['shards'], EXPECTED_UNITS):
    fold, seed = unit
    shard_id = f'p4b-f{fold}-s{seed}'
    assert entry == {
        'shard_id': shard_id, 'outer_fold': fold, 'training_seed': seed,
        'dataset_slug': f'thestonedape/task-aware-eeg2text-{shard_id}',
        'dataset_version': 1,
        'preserved_source_id': f'kaggle-dataset-thestonedape-task-aware-eeg2text-{shard_id}-version-1',
        'full_shard_manifest_sha256': entry['full_shard_manifest_sha256'],
        'verification_report_sha256': entry['verification_report_sha256'],
    }
    assert is_sha256(entry['full_shard_manifest_sha256']) and is_sha256(entry['verification_report_sha256'])
    assert digest(shard_roots[shard_id] / 'full_shard_manifest.json') == entry['full_shard_manifest_sha256']
    report_path = REGISTRY_ROOT / 'reports' / f'{shard_id}_verification_report.json'
    assert report_path.is_file() and not report_path.is_symlink() and digest(report_path) == entry['verification_report_sha256']
    report_by_shard[shard_id] = report_path
assert len(report_by_shard) == 15
print({'registry_sha256': REGISTRY_SHA256, 'verified_registry_entries': 15, 'attached_shard_datasets_v1': 15})


In [ ]:
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
command = [sys.executable, '-B', str(WORKTREE / 'evaluation/aggregate_task_segmented_full_shards.py')]
for fold, seed in EXPECTED_UNITS:
    shard_id = f'p4b-f{fold}-s{seed}'
    command.extend(['--shard', str(shard_roots[shard_id]), str(report_by_shard[shard_id])])
command.extend([
    '--launch-authorization-sha256', LAUNCH_SHA256,
    '--protocol-root', str(PROTOCOL_ROOT), '--schedule-root', str(SCHEDULE_ROOT),
    '--frozen-shard-registry', str(REGISTRY_PATH),
    '--expected-frozen-shard-registry-sha256', REGISTRY_SHA256,
    '--output-root', str(OUTPUT),
])
try:
    assert_clean_git()
    subprocess.run(command, check=True, cwd='/kaggle/working', env=test_env)
    assert_clean_git()
finally:
    if WORKTREE.exists():
        shutil.rmtree(WORKTREE)
    if ASKPASS.exists():
        ASKPASS.unlink()
manifest_path = OUTPUT / 'full_matrix_manifest.json'
integrity_path = OUTPUT / 'integrity_report.json'
decision_path = OUTPUT / 'scientific_decision.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
integrity = json.loads(integrity_path.read_text(encoding='utf-8'))
decision = json.loads(decision_path.read_text(encoding='utf-8'))
assert manifest['status'] == 'complete' and manifest['integrity_status'] == 'pass'
assert manifest['verified_shard_count'] == 15
assert manifest['launch_authorization_sha256'] == LAUNCH_SHA256
assert manifest['frozen_shard_registry_sha256'] == REGISTRY_SHA256
assert integrity['status'] == 'pass' and integrity['verified_shards'] == 15
transition = integrity['scientific_decision_permission_transition']
assert transition['partial_shard_scientific_decision_permitted'] is False
assert transition['complete_matrix_scientific_decision_permitted'] is True
assert decision['scientific_decision_permitted'] is True
assert decision['held_out_test_accessed'] is False
assert manifest['scientific_decision_status'] in {'pass', 'fail'}
assert decision['status'] == manifest['scientific_decision_status']
assert integrity['official_validation_used_for_confirmation'] is False
assert integrity['held_out_test_accessed'] is False
assert integrity['checkpoint_deserialized_by_aggregator'] is False
assert set(path.name for path in OUTPUT.iterdir()) == {
    'run_manifest.csv', 'confirmation_predictions.csv', 'integrity_report.json',
    'scientific_decision.json', 'full_matrix_manifest.json',
}
matrix_manifest_sha256 = digest(manifest_path)
assert not WORKTREE.exists() and not ASKPASS.exists()
print({
    'status': 'pass', 'integrity_status': 'pass',
    'scientific_decision_status': decision['status'],
    'continuation_decision': decision['continuation_decision'],
    'verified_shards': 15, 'full_matrix_manifest_sha256': matrix_manifest_sha256,
    'frozen_shard_registry_sha256': REGISTRY_SHA256,
    'partial_shard_scientific_decision_permitted': False,
    'complete_matrix_scientific_decision_permitted': True,
    'held_out_test_accessed': False, 'output_root': str(OUTPUT),
})
print('P4B COMPLETE-MATRIX AGGREGATION AND FROZEN SCIENTIFIC DECISION: PASS')


Preserve the aggregate output privately and record the printed full-matrix manifest SHA-256. The frozen `scientific_decision.json` determines whether the project continues (`continue_p4b`) or stops this P4b path permanently (`stop_p4b_permanently`).